In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer

In [2]:
train = pd.read_csv("spring2026_kaggle_linear_regression_challenge_train.csv")
test = pd.read_csv("spring2026_kaggle_linear_regression_challenge_test.csv")
sample = pd.read_csv("spring2026_sampleSubmission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample shape:", sample.shape)

print("\nTrain columns:")
print(train.columns)

print("\nTest columns:")
print(test.columns)

print("\nSample submission preview:")
print(sample.head())

FileNotFoundError: [Errno 2] No such file or directory: 'spring2026_kaggle_linear_regression_challenge_train.csv'

In [ ]:
TARGET = "target"
ID_COL = "Id"

# Feature columns: x0 through x14 plus Id
x_cols = [col for col in train.columns if col.startswith("x")]
feature_cols = x_cols + [ID_COL]

X_full = train[feature_cols]
y = train[TARGET]

X_test_full = test[feature_cols]
test_ids = test[ID_COL]

print("Feature columns:")
print(feature_cols)

print("\nX_full shape:", X_full.shape)
print("y shape:", y.shape)
print("X_test_full shape:", X_test_full.shape)
print("test_ids shape:", test_ids.shape)

In [ ]:
# Final blend weights
w_extra = 0.32
w_ridge = 0.68

# ExtraTrees component
extra_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("et", ExtraTreesRegressor(
        n_estimators=1000,
        max_depth=4,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

# Ridge component
ridge_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
    ("ridge", Ridge(alpha=3000))
])

In [ ]:
extra_model.fit(X_full, y)
ridge_model.fit(X_full, y)

print("Both models trained successfully.")

In [ ]:
pred_extra = extra_model.predict(X_test_full)
pred_ridge = ridge_model.predict(X_test_full)

final_predictions = (w_extra * pred_extra) + (w_ridge * pred_ridge)

print("ExtraTrees prediction shape:", pred_extra.shape)
print("Ridge prediction shape:", pred_ridge.shape)
print("Final prediction shape:", final_predictions.shape)

print("\nFinal prediction preview:")
print(final_predictions[:10])

In [ ]:
submission = pd.DataFrame({
    "Id": test_ids.values,
    "target": final_predictions
})

filename = "Obrycki_Pawel_submission_et32_ridge68_et1000_depth4_leaf20_ridge3000.csv"

submission.to_csv(filename, index=False)

print("Created:", filename)
submission.head()

In [ ]:
check = pd.read_csv("Obrycki_Pawel_submission_et32_ridge68_et1000_depth4_leaf20_ridge3000.csv")

print(check.head())
print("\nShape:", check.shape)
print("Columns:", check.columns.tolist())

if list(check.columns) == ["Id", "target"] and check.shape == (2500, 2):
    print("\nSubmission format is correct.")
else:
    print("\nWarning: submission format may be incorrect.")